# Olist: data sources and table grain

## Goal
Load the customers, orders and order-items tables, establish what one row means in each, and check the keys before any analysis.

## Setup
The raw CSV files are not committed. Run `python olist/download_data.py` from the repository root to fetch them into `olist/data/raw/`.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Project folder = the nearest parent that contains data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
DATA_DIR = ROOT / "data" / "raw"

csv_files = sorted(DATA_DIR.glob("*.csv"))
print("Local CSV files:", [p.name for p in csv_files])
if not csv_files:
    print("No Olist data found. Run olist/download_data.py first.")

Local CSV files: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## Source
- Dataset: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) on Kaggle
- License: CC BY-NC-SA 4.0
- Downloaded: 2026-09-29

## Customers, orders, order items
For each table: columns, shape, first rows, and whether the key is unique. Note that the customer key used in orders (`customer_id`) is not the identifier of a person (`customer_unique_id`).

In [2]:
# Grain (what one row represents):
#   customers: one customer record (a new one is created for every order)
#   orders:    one order
#   items:     one item within an order
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders    = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items     = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

In [3]:
print("--- customers: first 5 rows ---")
display(customers.head())

for name, df in [("customers", customers), ("orders", orders), ("items", items)]:
    print(f"--- {name}: {df.shape[0]} rows x {df.shape[1]} columns ---")
    print(df.dtypes, "\n")

--- customers: first 5 rows ---


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


--- customers: 99441 rows x 5 columns ---
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object 

--- orders: 99441 rows x 8 columns ---
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object 

--- items: 112650 rows x 7 columns ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object 



In [4]:
# Key checks: duplicates should be 0; then count distinct values

# customers: customer_id is the key
dup_customer_id = customers["customer_id"].duplicated().sum()
n_customer_id = customers["customer_id"].nunique()                    # one per order
n_customer_unique_id = customers["customer_unique_id"].nunique()      # one per person

# orders: order_id is the key
dup_order_id = orders["order_id"].duplicated().sum()
n_order_id = orders["order_id"].nunique()

# items: the key is (order_id, order_item_id)
dup_item_key = items.duplicated(subset=["order_id", "order_item_id"]).sum()
n_orders_in_items = items["order_id"].nunique()                       # fewer than in orders: some orders have no item rows

print("Duplicate customer_id:", dup_customer_id)
print("Distinct customer_id:", n_customer_id)
print("Distinct customer_unique_id:", n_customer_unique_id)

print("Duplicate order_id:", dup_order_id)
print("Distinct order_id:", n_order_id)

print("Duplicate item keys:", dup_item_key)
print("Orders present in items:", n_orders_in_items)

Duplicate customer_id: 0
Distinct customer_id: 99441
Distinct customer_unique_id: 96096
Duplicate order_id: 0
Distinct order_id: 99441
Duplicate item keys: 0
Orders present in items: 98666


## Checks
Is the join key between customers and orders unique? Is the relationship between orders and items one-to-many? Why can the row count change after a join?

In [5]:
# Check 1: how many people are behind the customer records?
print("customer_id (records):", customers["customer_id"].nunique())
print("customer_unique_id (people):", customers["customer_unique_id"].nunique())
print("Fewer people than records, so some customers ordered more than once.")

customer_id (records): 99441
customer_unique_id (people): 96096
Fewer people than records, so some customers ordered more than once.


In [6]:
# Check 2: orders to items is one-to-many
items_per_order = items.groupby("order_id").size()       # item rows per order
print("Most item rows in a single order:", items_per_order.max())

# Distribution: how many orders have 1 item row, 2 item rows, ...
dist = items_per_order.value_counts().sort_index().rename_axis("item_rows").reset_index(name="n_orders")
display(dist)

Most item rows in a single order: 21


,item_rows,n_orders
0,1,88863
1,2,7516
2,3,1322
3,4,505
4,5,204
5,6,198
6,7,22
7,8,8
8,9,3
9,10,8


In [7]:
# Check 3: row counts before and after a join
# Left join keeps every order and repeats it once per item row
orders_items_left = orders.merge(items, on="order_id", how="left")

compare = pd.DataFrame(
    [
        ("orders (before)", len(orders), orders["order_id"].nunique()),
        ("items (before)", len(items), items["order_id"].nunique()),
        ("orders left join items (after)", len(orders_items_left), orders_items_left["order_id"].nunique()),
    ],
    columns=["table", "rows", "distinct_orders"],
)
display(compare)
print("Rows increase after the join while the number of orders does not:")
print("after joining to items, count orders with nunique, never by counting rows.")

,table,rows,distinct_orders
0,orders (before),99441,99441
1,items (before),112650,98666
2,orders left join items (after),113425,99441


Rows increase after the join while the number of orders does not:
after joining to items, count orders with nunique, never by counting rows.


## Findings
- `customer_id` is one per order (99,441 rows, unique). People are identified by `customer_unique_id` (96,096), which is the column to use when counting customers.
- Orders have one row per order; items are one-to-many (up to 21 rows per order). After a left join there are 113,425 rows but still 99,441 orders, so orders must be counted with `nunique` after joining. Only 98,666 orders appear in items; the other 775 have no item rows.
- Amounts in the items table must be aggregated to one row per order before joining to orders, otherwise order-level values are counted more than once.

The analysis continues by topic: [02-monthly-orders](02-monthly-orders.ipynb), [03-monthly-gmv](03-monthly-gmv.ipynb), [04-orders-by-state](04-orders-by-state.ipynb), [05-delivery-time](05-delivery-time.ipynb), [06-monthly-orders-chart](06-monthly-orders-chart.ipynb), [07-delay-reviews](07-delay-reviews.ipynb), [08-category-gmv](08-category-gmv.ipynb).